# Domain 1: Agents and Workflows
**Exam**: Claude Certified Developer – Foundations (CCDV-F)
**Weighting**: 14.7% of exam
**Depth**: deep

## Domain Overview

This domain covers agent architecture principles (workflow vs. agent, manager/subagent
hierarchies), constructing agents with the Claude Agent SDK (custom loops, hooks, hosting models),
and agent design patterns/frameworks (tool-use loops, memory, LangGraph/PydanticAI-style
abstractions).

**Why it matters**: developers are expected to know not just how to call the API, but how to build
and choose between agent architectures for real multi-step tasks.

**Key concepts to know**: workflow vs. agent decision criteria, the Claude Agent SDK's agentic loop
and hooks, subagent delegation, and common agentic design patterns.

In [ ]:
# Setup: a small helper so every cell in this notebook runs end-to-end,
# whether or not you have an ANTHROPIC_API_KEY configured.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    """Calls the real Claude API if ANTHROPIC_API_KEY is set; otherwise returns a
    simulated response so this notebook is runnable without API access."""
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception as e:
        return (f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY and `pip install anthropic` "
                f"to call the real API]\nPrompt received ({len(prompt)} chars): {prompt[:150]}...")

print("call_claude() ready. Set ANTHROPIC_API_KEY as an environment variable to hit the real API.")

## Key Concepts

### Agent Architecture: workflow vs. agent decision criteria

Use a **workflow** (fixed, code-defined sequence of LLM calls) when the steps and their
order are known in advance. Use an **agent** (the model decides the next action based on context)
when the path varies per input. Manager/supervisor hierarchies delegate specialized work to
subagents, which improves task execution by isolating context and specialization.

In [ ]:
def workflow_or_agent(steps_known_in_advance: bool) -> str:
    return "workflow" if steps_known_in_advance else "agent"

print(workflow_or_agent(True))   # e.g., a fixed 3-step document pipeline
print(workflow_or_agent(False))  # e.g., open-ended codebase investigation

### Agent Construction with Claude: Agent SDK, custom loops, hooks, hosting

The Claude Agent SDK provides agent loop scaffolding, subagent spawning, and hooks (e.g.
`PostToolUse`) for deterministic actions. Custom loops/harnesses give full control when the SDK's
opinionated structure doesn't fit. Hosting can be self-managed (you run the loop) or Anthropic-hosted
(managed infrastructure) — a deployment tradeoff of control vs. operational overhead.

In [ ]:
def simple_agentic_loop(send_request, tools, max_iterations=8):
    messages = []
    for _ in range(max_iterations):
        response = send_request(messages, tools)
        if response["stop_reason"] == "end_turn":
            return response
        for tool_call in response.get("tool_calls", []):
            messages.append({"role": "tool_result", "content": f"result for {tool_call}"})
    raise RuntimeError("did not reach end_turn - investigate, don't just raise the cap")

def fake_send_request(messages, tools):
    return {"stop_reason": "end_turn", "tool_calls": []} if len(messages) >= 2 else {"stop_reason": "tool_use", "tool_calls": ["lookup"]}

print(simple_agentic_loop(fake_send_request, tools=["lookup"]))

### Agent Patterns and Frameworks: tool-use loops, subagents, memory, abstraction frameworks

Common patterns: the tool-use loop (as above), sub-agents for context isolation, and
memory (persisting facts across turns/sessions). Agentic abstraction frameworks (Strands, LangGraph,
PydanticAI) provide graph- or state-machine-based structuring for complex multi-step workflows,
trading some flexibility for more explicit control flow.

In [ ]:
class SimpleMemory:
    def __init__(self):
        self.facts = {}
    def remember(self, key, value):
        self.facts[key] = value
    def recall(self, key):
        return self.facts.get(key)

memory = SimpleMemory()
memory.remember("customer_tier", "enterprise")
print(memory.recall("customer_tier"))

# A LangGraph-style state machine sketch (conceptual, not the real library):
graph_nodes = {"classify": "route to specialist", "specialist": "produce answer", "review": "final check"}
print(graph_nodes)

## Mini Hands-On Project

**Covers**: all Domain 1 sub-objectives.

**Scenario**: Build a minimal ticket-triage agent that decides per-ticket whether to use a fixed
workflow or agentic reasoning, with a simple memory of prior classifications.

### Step 1: Decide workflow vs. agent per ticket type

In [ ]:
ticket_types = {"password reset": True, "ambiguous billing dispute": False}
for t, known in ticket_types.items():
    print(t, "->", workflow_or_agent(known))

### Step 2: Run the agentic loop for the ambiguous case

In [ ]:
print(simple_agentic_loop(fake_send_request, tools=["lookup_billing"]))

### Step 3: Persist a classification fact to memory

In [ ]:
memory.remember("last_ticket_type", "billing dispute")
print(memory.recall("last_ticket_type"))

## Lab Exercises

### Exercise 1: Workflow vs. agent classification

**Objective**: Classify 5 new tasks as workflow or agent-appropriate.

**Produce**: 5 classifications with reasoning.

In [ ]:
tasks = ["Nightly data export (same 3 steps every time)", "Open-ended customer complaint investigation",
         "Fixed onboarding email sequence", "Debugging an unfamiliar production incident", "Ad-hoc research question"]
# TODO: classify each using workflow_or_agent-style reasoning
for t in tasks:
    pass

# Expected output: Fixed, predictable tasks classified as workflow; open-ended/variable tasks classified as agent.

In [ ]:
# STRETCH GOAL: for one "agent" task, sketch a 2-subagent manager/supervisor hierarchy for it.
subagent_design = {}  # TODO
print(subagent_design)

# Expected output: A plausible 2-subagent design (e.g., investigator + summarizer) with a manager coordinating them.

### Exercise 2: Fix a broken agentic loop

**Objective**: Fix a version of simple_agentic_loop that doesn't correctly accumulate tool results.

**Produce**: A corrected loop.

In [ ]:
def broken_loop(send_request, tools, max_iterations=8):
    for _ in range(max_iterations):
        response = send_request([], tools)  # BUG: always sends an empty message list, losing all history
        if response["stop_reason"] == "end_turn":
            return response
    raise RuntimeError("no end_turn")

# TODO: write fixed_loop that accumulates messages correctly across iterations (like simple_agentic_loop)
def fixed_loop(send_request, tools, max_iterations=8):
    pass

print(fixed_loop(fake_send_request, tools=["lookup"]))

# Expected output: fixed_loop correctly reaches end_turn by accumulating tool results across iterations, unlike broken_loop.

In [ ]:
# STRETCH GOAL: add a hook-like callback that logs every tool call before it executes.
def fixed_loop_with_logging(send_request, tools, on_tool_call=None, max_iterations=8):
    pass
fixed_loop_with_logging(fake_send_request, ["lookup"], on_tool_call=lambda t: print("calling:", t))

# Expected output: Logged output showing each tool call before execution.

### Exercise 3: Memory-augmented agent

**Objective**: Extend SimpleMemory to support a TTL (facts expire after N 'turns').

**Produce**: An updated memory class with expiring facts.

In [ ]:
class TTLMemory(SimpleMemory):
    def __init__(self):
        super().__init__()
        self.turn = 0
        self.expirations = {}
    def remember(self, key, value, ttl_turns=None):
        # TODO: store value and, if ttl_turns given, self.expirations[key] = self.turn + ttl_turns
        pass
    def advance_turn(self):
        self.turn += 1
        # TODO: remove any keys whose expiration has passed
    def recall(self, key):
        return self.facts.get(key)

m = TTLMemory()
m.remember("temp_flag", True, ttl_turns=1)
m.advance_turn()
m.advance_turn()
print(m.recall("temp_flag"))  # should be None after expiring

# Expected output: temp_flag is correctly removed after its TTL elapses.

In [ ]:
# STRETCH GOAL: add a `remember_permanent` convenience method with no TTL, and verify it never expires.

# Expected output: A permanent fact that survives multiple advance_turn() calls.

### Exercise 4: Design a manager/subagent hierarchy

**Objective**: Design a 3-subagent hierarchy for a code-migration agent.

**Produce**: A manager + 3 subagent role design with responsibilities.

In [ ]:
# TODO: design roles for a manager and 3 subagents (e.g., planner, executor, verifier) for
# migrating a codebase from one framework to another
hierarchy = {
    "manager": "...",
    "subagents": {
        # "planner": "...",
    },
}
print(hierarchy)

# Expected output: A sensible manager + 3 distinctly-scoped subagent roles covering planning, execution, and verification.

In [ ]:
# STRETCH GOAL: identify which subagent should own context isolation (i.e., which one benefits
# most from NOT inheriting the manager's full history) and why.
isolation_choice = "..."  # TODO

# Expected output: A reasoned choice, likely the verifier, to avoid bias from the executor's reasoning trail.

### Exercise 5: Framework tradeoff analysis

**Objective**: Compare a custom agentic loop vs. a graph-based framework for one scenario.

**Produce**: A short tradeoff analysis and a recommendation.

In [ ]:
scenario = "A support agent with 4 possible states (triage, resolve, escalate, close) and clear transition rules between them."

# TODO: analyze whether a custom loop or a graph-based framework (like LangGraph) fits better, and why
recommendation = "..."  # TODO
reasoning = "..."  # TODO
print(recommendation, reasoning)

# Expected output: Recommends a graph-based framework given the clear, enumerable states/transitions, with sound reasoning.

In [ ]:
# STRETCH GOAL: sketch the state graph (nodes and edges) for this scenario as a Python dict.
state_graph = {}  # TODO: e.g. {"triage": ["resolve", "escalate"], "resolve": ["close"], ...}
print(state_graph)

# Expected output: A plausible state graph covering all 4 states and their valid transitions.

## Self-Assessment: Agents and Workflows

Answer each question, then check the Answers cell below.


**Q1.** What is the key decision criterion for choosing a workflow over an agent architecture?

**Q2.** What does a `PostToolUse` hook in the Claude Agent SDK allow you to do?

**Q3.** What is the tradeoff between self-hosting an agent loop vs. using a managed/hosted deployment model?

**Q4.** When would a graph-based agentic framework (like LangGraph) be preferable to a custom loop?

**Q5.** Why might a subagent benefit from NOT inheriting the manager agent's full conversation history?

## Answers


**A1.** Whether the steps and their order are known and predictable in advance (workflow) vs. needing to vary per input (agent).

**A2.** It lets you intercept and transform/act on tool results deterministically after a tool call completes, before the model sees them.

**A3.** Self-hosting gives more control but more operational overhead; managed hosting reduces overhead but gives up some control/customization.

**A4.** When the task has clearly enumerable states and transitions, making explicit state-machine structure more maintainable than ad-hoc loop logic.

**A5.** To avoid the subagent's judgment being biased or anchored by the manager's (or another agent's) prior reasoning, and to keep its context focused and small.

---
**Next**: [Domain 2: Applications and Integration](./02-applications-and-integration.ipynb)
**Back**: [README](./README.md)